# Interaktive Tourenkarte

Dieses Notebook lädt die CSV, erstellt eine interaktive Karte pro Tag/LKW/Tour und exportiert zusätzlich eine HTML-Datei.


In [1]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from ipywidgets import interact, Dropdown
from IPython.display import display
from pathlib import Path

CSV_PATH = 'istdaten_prc_clean.csv'
OUT_HTML = Path('output/tourenkarte.html')
OUT_HTML.parent.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH, encoding='utf-8-sig', low_memory=False)
for col in ['Longitude', 'Latitude']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['Zeitstempel_Fahrzeug'] = pd.to_datetime(df['Zeitstempel_Fahrzeug'], errors='coerce')
df = df.dropna(subset=['Longitude', 'Latitude', 'Zeitstempel_Fahrzeug']).copy()
df = df.sort_values(['LKW_KENNZ', 'Zeitstempel_Fahrzeug', 'PositionsID'])
df['date'] = df['Zeitstempel_Fahrzeug'].dt.date.astype(str)
df['time'] = df['Zeitstempel_Fahrzeug'].dt.strftime('%H:%M:%S')
dates = sorted(df['date'].dropna().unique().tolist())
dates[:5], len(dates)


(['2026-03-01', '2026-03-02', '2026-03-03', '2026-03-04', '2026-03-05'], 31)

In [2]:
def filtered_values(date_value, vehicle_value=None):
    dff = df[df['date'] == date_value].copy()
    vehicles = sorted(dff['LKW_KENNZ'].dropna().unique().tolist())
    return vehicles

def make_map(date_value, vehicle_value=None):
    dff = df[df['date'] == date_value].copy()

    vehicles = sorted(dff['LKW_KENNZ'].dropna().unique().tolist())
    if vehicle_value is None or vehicle_value not in vehicles:
        vehicle_value = vehicles[0] if vehicles else None

    if vehicle_value is not None:
        dff = dff[dff['LKW_KENNZ'] == vehicle_value].copy()

    if dff.empty:
        center = [53.55, 9.99]
    else:
        center = [dff['Latitude'].mean(), dff['Longitude'].mean()]

    m = folium.Map(location=center, zoom_start=11, tiles='OpenStreetMap')

    if not dff.empty:
        g = dff.sort_values('Zeitstempel_Fahrzeug')
        pts = g[['Latitude', 'Longitude']].to_numpy().tolist()

        if len(pts) >= 2:
            folium.PolyLine(pts, color='blue', weight=4, opacity=0.8).add_to(m)

        cluster = MarkerCluster().add_to(m)
        for _, r in g.iterrows():
            popup = (
                f"{r.get('LKW_KENNZ','')}"
                f"<br>GPS: {r.get('Zeitstempel_Fahrzeug','')}"
                f"<br>Typ: {r.get('msg_typ','')}"
                f"<br>Station: {r.get('StationID','')}"
            )
            folium.CircleMarker(
                location=[r['Latitude'], r['Longitude']],
                radius=4,
                color='red',
                fill=True,
                fill_opacity=0.9,
                tooltip=r['time'],
                popup=folium.Popup(popup, max_width=320),
            ).add_to(cluster)

    return m

def export_html(date_value, vehicle_value=None, path=OUT_HTML):
    m = make_map(date_value, vehicle_value)
    m.save(str(path))
    return path

In [3]:
date_dropdown = Dropdown(options=dates, value=dates[0] if dates else None, description='Tag')
vehicle_dropdown = Dropdown(description='LKW')

def refresh_options(*args):
    vehicles = filtered_values(date_dropdown.value, vehicle_dropdown.value)
    vehicle_dropdown.options = vehicles
    if vehicle_dropdown.value not in vehicles:
        vehicle_dropdown.value = vehicles[0] if vehicles else None

refresh_options()

def show_map(date_value, vehicle_value):
    m = make_map(date_value, vehicle_value)
    display(m)
    export_html(date_value, vehicle_value)
    print(f'HTML export: {OUT_HTML}')

date_dropdown.observe(lambda change: refresh_options(), names='value')
vehicle_dropdown.observe(lambda change: refresh_options(), names='value')

interact(show_map, date_value=date_dropdown, vehicle_value=vehicle_dropdown);

interactive(children=(Dropdown(description='Tag', options=('2026-03-01', '2026-03-02', '2026-03-03', '2026-03-…